# Седмица 10: WebSockets в JavaScript

Двупосочна комуникация в реално време между браузър и сървър. Тази тетрадка обхваща както Native WebSocket API, така и библиотеката Socket.IO.

**Забележка:** Примерите с WebSocket изискват сървър. За всеки раздел е предоставен сървърен код — стартирайте го с Node.js.


## Съдържание

### Част 1: Native WebSockets
1. Основи на WebSocket
2. Клиентска връзка и събития
3. Изпращане и получаване на съобщения
4. WebSocket сървър (Node.js)
5. Бинарни данни
6. Шаблон за повторно свързване

### Част 2: Socket.IO
7. Въведение в Socket.IO
8. Настройка на Socket.IO сървър
9. Socket.IO клиент
10. Събития и потвърждения
11. Стаи и пространства от имена
12. Разпращане (Broadcasting)

### Част 3: Сравнение и шаблони
13. Native vs Socket.IO
14. Пример за чат в реално време


---
## Част 1: Native WebSockets
---


### 1. Основи на WebSocket

**Цел:** WebSockets осигуряват пълнодуплексни комуникационни канали през една TCP връзка. За разлика от HTTP (заявка-отговор), WebSockets позволяват на клиента и сървъра да изпращат съобщения по всяко време.

**Ключови концепции:**
- **Пълен дуплекс:** И двете страни могат да изпращат/получават едновременно
- **Постоянна връзка:** Без повторни handshakes
- **Ниска латентност:** Без HTTP overhead след връзката
- **Протокол:** `ws://` (некриптиран) или `wss://` (криптиран/TLS)

[MDN: WebSocket](https://developer.mozilla.org/en-US/docs/Web/API/WebSocket) | [RFC 6455](https://datatracker.ietf.org/doc/html/rfc6455)


In [ ]:
// Състояния на WebSocket връзката
console.log('WebSocket състояния:');
console.log('CONNECTING:', WebSocket.CONNECTING); // 0
console.log('OPEN:', WebSocket.OPEN);             // 1
console.log('CLOSING:', WebSocket.CLOSING);       // 2
console.log('CLOSED:', WebSocket.CLOSED);         // 3


### 2. Клиентска връзка и събития

**Цел:** Установяване на WebSocket връзка и обработка на събития от жизнения цикъл.

**Събития:**
- `open` - Връзката е установена
- `message` - Получени данни от сървъра
- `error` - Възникнала е грешка
- `close` - Връзката е затворена

[MDN: WebSocket Events](https://developer.mozilla.org/en-US/docs/Web/API/WebSocket#events)


In [ ]:
// Браузърен клиент - WebSocket връзка
// Изпълнете в браузърната конзола след стартиране на сървъра

const ws = new WebSocket('ws://localhost:8080');

// Връзката е отворена
ws.addEventListener('open', (event) => {
  console.log('✅ Свързан със сървъра');
  console.log('Ready state:', ws.readyState); // 1 = OPEN
});

// Слушане за съобщения
ws.addEventListener('message', (event) => {
  console.log('📨 Съобщение от сървъра:', event.data);
});

// Грешка във връзката
ws.addEventListener('error', (event) => {
  console.error('❌ WebSocket грешка:', event);
});

// Връзката е затворена
ws.addEventListener('close', (event) => {
  console.log('🔌 Връзката е затворена');
  console.log('Код:', event.code);
  console.log('Причина:', event.reason);
  console.log('Чисто затваряне:', event.wasClean);
});


### 3. Изпращане и получаване на съобщения

**Цел:** Обмен на текстови и JSON данни между клиент и сървър.

[MDN: WebSocket.send()](https://developer.mozilla.org/en-US/docs/Web/API/WebSocket/send)


In [ ]:
// Изпращане на съобщения (изпълнете след като връзката е отворена)

// Изпращане на обикновен текст
ws.send('Здравей, сървър!');

// Изпращане на JSON данни
const message = {
  type: 'chat',
  user: 'Алиса',
  content: 'Здравейте на всички!',
  timestamp: Date.now()
};
ws.send(JSON.stringify(message));

// Проверка на буфера
console.log('Буферирани байтове:', ws.bufferedAmount);


In [ ]:
// Получаване и парсване на JSON съобщения

ws.addEventListener('message', (event) => {
  try {
    const data = JSON.parse(event.data);
    
    switch (data.type) {
      case 'chat':
        console.log(`[${data.user}]: ${data.content}`);
        break;
      case 'notification':
        console.log('📢', data.message);
        break;
      case 'error':
        console.error('Сървърна грешка:', data.message);
        break;
      default:
        console.log('Неизвестен тип съобщение:', data);
    }
  } catch (e) {
    // Обикновено текстово съобщение
    console.log('Текстово съобщение:', event.data);
  }
});


### 4. WebSocket сървър (Node.js)

**Цел:** Създаване на WebSocket сървър с библиотеката `ws`.

**Настройка:**
```bash
npm install ws
```

[ws npm пакет](https://www.npmjs.com/package/ws)


In [ ]:
// server.js - Основен WebSocket сървър
// Стартирайте с: node server.js

const { WebSocketServer } = require('ws');

const wss = new WebSocketServer({ port: 8080 });

console.log('WebSocket сървърът работи на ws://localhost:8080');

wss.on('connection', (ws, req) => {
  const clientIP = req.socket.remoteAddress;
  console.log(`Нов клиент се свърза от ${clientIP}`);
  
  // Изпращане на приветствено съобщение
  ws.send(JSON.stringify({
    type: 'notification',
    message: 'Добре дошли на сървъра!'
  }));
  
  // Обработка на входящи съобщения
  ws.on('message', (data) => {
    console.log('Получено:', data.toString());
    
    // Връщане на ехо към клиента
    ws.send(`Ехо: ${data}`);
    
    // Разпращане до всички клиенти
    wss.clients.forEach((client) => {
      if (client !== ws && client.readyState === 1) {
        client.send(data.toString());
      }
    });
  });
  
  // Обработка на прекъсване на връзката
  ws.on('close', () => {
    console.log('Клиентът се изключи');
  });
  
  // Обработка на грешки
  ws.on('error', (error) => {
    console.error('WebSocket грешка:', error);
  });
});


### 5. Бинарни данни

**Цел:** Изпращане и получаване на бинарни данни (файлове, изображения, аудио) през WebSockets.

[MDN: Blob](https://developer.mozilla.org/en-US/docs/Web/API/Blob) | [MDN: ArrayBuffer](https://developer.mozilla.org/en-US/docs/Web/JavaScript/Reference/Global_Objects/ArrayBuffer)


In [ ]:
// Изпращане на бинарни данни от клиента

// Задаване на бинарен тип (по подразбиране е 'blob')
ws.binaryType = 'arraybuffer'; // или 'blob'

// Изпращане на ArrayBuffer
const buffer = new ArrayBuffer(8);
const view = new Uint8Array(buffer);
view.set([72, 101, 108, 108, 111, 33, 33, 33]); // "Hello!!!"
ws.send(buffer);

// Изпращане на Blob (качване на файл)
const blob = new Blob(['Здравей от Blob!'], { type: 'text/plain' });
ws.send(blob);

// Получаване на бинарни данни
ws.addEventListener('message', (event) => {
  if (event.data instanceof ArrayBuffer) {
    const text = new TextDecoder().decode(event.data);
    console.log('Бинарни (ArrayBuffer):', text);
  } else if (event.data instanceof Blob) {
    event.data.text().then(text => {
      console.log('Бинарни (Blob):', text);
    });
  }
});


### 6. Шаблон за повторно свързване

**Цел:** Автоматично повторно свързване при прекъсване — от съществено значение за продуктови приложения.


In [ ]:
// Устойчив WebSocket клиент с автоматично повторно свързване

class ReconnectingWebSocket {
  constructor(url, options = {}) {
    this.url = url;
    this.maxRetries = options.maxRetries || 10;
    this.retryDelay = options.retryDelay || 1000;
    this.maxDelay = options.maxDelay || 30000;
    this.retries = 0;
    this.ws = null;
    this.listeners = { open: [], message: [], close: [], error: [] };
    
    this.connect();
  }
  
  connect() {
    console.log(`Свързване към ${this.url}...`);
    this.ws = new WebSocket(this.url);
    
    this.ws.addEventListener('open', (e) => {
      console.log('✅ Свързан');
      this.retries = 0;
      this.emit('open', e);
    });
    
    this.ws.addEventListener('message', (e) => {
      this.emit('message', e);
    });
    
    this.ws.addEventListener('close', (e) => {
      this.emit('close', e);
      this.reconnect();
    });
    
    this.ws.addEventListener('error', (e) => {
      this.emit('error', e);
    });
  }
  
  reconnect() {
    if (this.retries >= this.maxRetries) {
      console.error('Достигнат максимален брой опити. Отказвам.');
      return;
    }
    
    // Експоненциално нарастване на забавянето
    const delay = Math.min(
      this.retryDelay * Math.pow(2, this.retries),
      this.maxDelay
    );
    
    console.log(`Повторно свързване след ${delay}ms (опит ${this.retries + 1})`);
    this.retries++;
    
    setTimeout(() => this.connect(), delay);
  }
  
  send(data) {
    if (this.ws.readyState === WebSocket.OPEN) {
      this.ws.send(data);
    } else {
      console.warn('WebSocket не е отворен. Съобщението е на опашка.');
    }
  }
  
  on(event, callback) {
    this.listeners[event]?.push(callback);
  }
  
  emit(event, data) {
    this.listeners[event]?.forEach(cb => cb(data));
  }
  
  close() {
    this.maxRetries = 0;
    this.ws.close();
  }
}

// Употреба
const socket = new ReconnectingWebSocket('ws://localhost:8080');

socket.on('message', (e) => {
  console.log('Получено:', e.data);
});

socket.send('Здравей с автоматично повторно свързване!');


### 7. Въведение в Socket.IO

**Цел:** Socket.IO е библиотека, която позволява двупосочна комуникация в реално време. Тя надгражда WebSockets, но добавя функции като автоматично повторно свързване, стаи, пространства от имена и резервни транспорти.

**Ключови функции:**
- Автоматично повторно свързване
- Буфериране на пакети при прекъсване
- Потвърждения (шаблон заявка-отговор)
- Стаи и пространства от имена
- Резервен вариант към HTTP long-polling
- Разпращане (Broadcasting)

**Забележка:** Socket.IO НЕ е WebSocket имплементация. Socket.IO клиенти не могат да се свързват с обикновени WebSocket сървъри и обратно.

[Socket.IO документация](https://socket.io/docs/v4/)


---
## Част 2: Socket.IO
---


### 8. Настройка на Socket.IO сървър

**Настройка:**
```bash
npm install socket.io
```

[Socket.IO Server API](https://socket.io/docs/v4/server-api/)


In [ ]:
// socket-server.js - Socket.IO сървър
// Стартирайте с: node socket-server.js

const { Server } = require('socket.io');
const http = require('http');

const httpServer = http.createServer();

const io = new Server(httpServer, {
  cors: {
    origin: '*',
    methods: ['GET', 'POST']
  }
});

io.on('connection', (socket) => {
  console.log(`Клиент се свърза: ${socket.id}`);
  
  socket.emit('welcome', {
    message: 'Добре дошли в Socket.IO сървъра!',
    id: socket.id
  });
  
  socket.on('chat message', (data) => {
    console.log('Получено съобщение:', data);
    socket.broadcast.emit('chat message', {
      ...data,
      from: socket.id
    });
  });
  
  socket.on('disconnect', (reason) => {
    console.log(`Клиент се изключи: ${socket.id}, причина: ${reason}`);
  });
});

const PORT = 3000;
httpServer.listen(PORT, () => {
  console.log(`Socket.IO сървър работи на http://localhost:${PORT}`);
});


In [ ]:
// Express + Socket.IO сървър
// npm install express socket.io

const express = require('express');
const { createServer } = require('http');
const { Server } = require('socket.io');

const app = express();
const httpServer = createServer(app);
const io = new Server(httpServer);

// Сервиране на статични файлове
app.use(express.static('public'));

// REST endpoint
app.get('/api/status', (req, res) => {
  res.json({
    status: 'online',
    clients: io.sockets.sockets.size
  });
});

// Socket.IO събития
io.on('connection', (socket) => {
  console.log('Потребител се свърза:', socket.id);
  
  socket.on('disconnect', () => {
    console.log('Потребител се изключи:', socket.id);
  });
});

httpServer.listen(3000, () => {
  console.log('Express + Socket.IO на http://localhost:3000');
});


### 9. Socket.IO клиент

**Браузърна настройка:**
```html
<script src="https://cdn.socket.io/4.7.4/socket.io.min.js"></script>
```

**Node.js настройка:**
```bash
npm install socket.io-client
```

[Socket.IO Client API](https://socket.io/docs/v4/client-api/)


In [ ]:
// Браузърен клиент
// Първо включете socket.io клиентската библиотека

const socket = io('http://localhost:3000');

// Събития за връзката
socket.on('connect', () => {
  console.log('✅ Свързан с ID:', socket.id);
});

socket.on('disconnect', (reason) => {
  console.log('🔌 Изключен:', reason);
  
  if (reason === 'io server disconnect') {
    socket.connect();
  }
});

socket.on('connect_error', (error) => {
  console.error('❌ Грешка при свързване:', error.message);
});

socket.on('welcome', (data) => {
  console.log('📨 Приветствено съобщение:', data.message);
});


### 10. Събития и потвърждения

**Цел:** Персонализирани събития позволяват семантични съобщения. Потвържденията позволяват шаблон заявка-отговор.

[Socket.IO Emitting Events](https://socket.io/docs/v4/emitting-events/)


In [ ]:
// Node.js клиент
// const { io } = require('socket.io-client');

const socket = io('http://localhost:3000', {
  reconnection: true,
  reconnectionAttempts: 5,
  reconnectionDelay: 1000,
  timeout: 10000,
  autoConnect: true
});

socket.on('connect', () => {
  console.log('Свързан със сървъра');
  
  // Изпращане на съобщение
  socket.emit('chat message', {
    user: 'NodeClient',
    text: 'Здравей от Node.js!'
  });
});


In [ ]:
// Персонализирани събития - Клиент

// Изпращане на събитие с данни
socket.emit('chat message', {
  user: 'Алиса',
  text: 'Здравейте на всички!',
  timestamp: Date.now()
});

// Изпращане с потвърждение (callback)
socket.emit('save message', { text: 'Важно!' }, (response) => {
  if (response.status === 'ok') {
    console.log('Съобщението е запазено с ID:', response.id);
  } else {
    console.error('Неуспешно запазване:', response.error);
  }
});

// Слушане за събития
socket.on('chat message', (data) => {
  console.log(`[${data.user}]: ${data.text}`);
});

socket.on('user joined', (data) => {
  console.log(`${data.user} се присъедини към чата`);
});

socket.on('user left', (data) => {
  console.log(`${data.user} напусна чата`);
});


In [ ]:
// Персонализирани събития - Сървър

io.on('connection', (socket) => {
  // Обработка на събития с потвърждение
  socket.on('save message', (data, callback) => {
    try {
      const id = Math.random().toString(36).substr(2, 9);
      callback({ status: 'ok', id: id });
    } catch (error) {
      callback({ status: 'error', error: error.message });
    }
  });
  
  // Изпращане до единичен клиент
  socket.emit('private message', { text: 'Само вие виждате това' });
  
  // Изпращане до всички клиенти освен подателя
  socket.broadcast.emit('user joined', { user: socket.id });
  
  // Изпращане до всички клиенти включително подателя
  io.emit('announcement', { text: 'Нов потребител се свърза!' });
});


### 11. Стаи и пространства от имена

**Цел:** Организиране на връзките в логически групи.
- **Стаи (Rooms):** Подгрупи в рамките на namespace за целево разпращане
- **Пространства от имена (Namespaces):** Отделни комуникационни канали върху една връзка

[Socket.IO Rooms](https://socket.io/docs/v4/rooms/) | [Socket.IO Namespaces](https://socket.io/docs/v4/namespaces/)


In [ ]:
// Стаи - Сървър

io.on('connection', (socket) => {
  // Присъединяване към стая
  socket.on('join room', (roomName) => {
    socket.join(roomName);
    console.log(`${socket.id} се присъедини към стая: ${roomName}`);
    
    socket.to(roomName).emit('user joined', {
      user: socket.id,
      room: roomName
    });
  });
  
  // Напускане на стая
  socket.on('leave room', (roomName) => {
    socket.leave(roomName);
    socket.to(roomName).emit('user left', { user: socket.id });
  });
  
  // Изпращане на съобщение до стая
  socket.on('room message', ({ room, message }) => {
    io.to(room).emit('chat message', {
      from: socket.id,
      text: message
    });
  });
  
  // Получаване на стаите, в които е сокетът
  console.log('Стаи на сокета:', socket.rooms);
});


In [ ]:
// Стаи - Клиент

// Присъединяване към стая
socket.emit('join room', 'general');
socket.emit('join room', 'javascript');

// Изпращане на съобщение до стая
socket.emit('room message', {
  room: 'javascript',
  message: 'Някой тук знае TypeScript?'
});

// Напускане на стая
socket.emit('leave room', 'general');


In [ ]:
// Пространства от имена - Сървър

const { Server } = require('socket.io');
const io = new Server(httpServer);

// Пространство по подразбиране
io.on('connection', (socket) => {
  console.log('Свързан към default namespace');
});

// Персонализирано пространство за чат
const chatNamespace = io.of('/chat');
chatNamespace.on('connection', (socket) => {
  console.log('Свързан към /chat namespace');
  
  socket.on('message', (data) => {
    chatNamespace.emit('message', data);
  });
});

// Персонализирано пространство за известия
const notifyNamespace = io.of('/notifications');
notifyNamespace.on('connection', (socket) => {
  console.log('Свързан към /notifications namespace');
  
  socket.on('subscribe', (userId) => {
    socket.join(`user:${userId}`);
  });
});

// Изпращане на известие до конкретен потребител
function notifyUser(userId, notification) {
  notifyNamespace.to(`user:${userId}`).emit('notification', notification);
}


In [ ]:
// Пространства от имена - Клиент

// Свързване към default namespace
const defaultSocket = io('http://localhost:3000');

// Свързване към chat namespace
const chatSocket = io('http://localhost:3000/chat');
chatSocket.on('connect', () => {
  console.log('Свързан към чат');
  chatSocket.emit('message', { text: 'Здравей, чат!' });
});

// Свързване към notifications namespace
const notifySocket = io('http://localhost:3000/notifications');
notifySocket.on('connect', () => {
  console.log('Свързан към известия');
  notifySocket.emit('subscribe', 'user123');
});

notifySocket.on('notification', (data) => {
  console.log('🔔 Известие:', data);
});


### 12. Разпращане (Broadcasting)

**Цел:** Ефективно изпращане на съобщения до множество клиенти.

[Socket.IO Broadcasting](https://socket.io/docs/v4/broadcasting-events/)


In [ ]:
// Шаблони за разпращане - Сървър

io.on('connection', (socket) => {
  
  // 1. Изпращане само до подателя
  socket.emit('private', 'Само вие получавате това');
  
  // 2. Изпращане до всички освен подателя
  socket.broadcast.emit('broadcast', 'Всички освен подателя');
  
  // 3. Изпращане до всички включително подателя
  io.emit('global', 'Всички получават това');
  
  // 4. Изпращане до конкретна стая (без подателя)
  socket.to('room1').emit('room message', 'Само до room1');
  
  // 5. Изпращане до конкретна стая (с подателя)
  io.to('room1').emit('room message', 'До room1 с подателя');
  
  // 6. Изпращане до няколко стаи
  io.to('room1').to('room2').emit('multi room', 'До стаи 1 и 2');
  
  // 7. Изпращане до конкретен сокет по ID
  io.to(socketId).emit('direct', 'Директно съобщение до конкретен сокет');
  
  // 8. Изпращане до всички освен определени сокети
  socket.broadcast.except('room1').emit('filtered', 'Не до room1');
});


---
## Част 3: Сравнение и шаблони
---


### 13. Native WebSocket vs Socket.IO

| Функция | Native WebSocket | Socket.IO |
|---------|-----------------|----------|
| **Протокол** | Стандартен WebSocket (RFC 6455) | Персонализиран протокол върху WebSocket/HTTP |
| **Повторно свързване** | Ръчна имплементация | Автоматично с backoff |
| **Резервен вариант** | Няма | HTTP long-polling |
| **Събития** | Само `message` събитие | Персонализирани именувани събития |
| **Потвърждения** | Ръчна имплементация | Вградени callbacks |
| **Стаи** | Ръчна имплементация | Вградена поддръжка |
| **Бинарни данни** | ArrayBuffer/Blob | Автоматично разпознаване |
| **Мултиплексиране** | Една връзка на endpoint | Namespaces върху една връзка |
| **Размер на bundle** | Native (0kb) | ~50kb минифициран |
| **Съвместимост** | Само WebSocket клиенти | Само Socket.IO клиенти |

**Кога да използвате Native WebSocket:**
- Прости нужди за реално време
- Нужда от минимален размер на bundle
- Взаимодействие с други WebSocket имплементации
- Пълен контрол над протокола

**Кога да използвате Socket.IO:**
- Сложни приложения в реално време
- Нужда от стаи/пространства от имена
- Важна е резервната опция за браузъри
- Приоритет е бързата разработка


### 14. Пример за чат в реално време

**Цел:** Пълен пример за просто чат приложение с Native WebSocket и Socket.IO имплементации.


In [ ]:
// chat-server-native.js - Native WebSocket чат сървър
// npm install ws
// Стартирайте: node chat-server-native.js

const { WebSocketServer } = require('ws');

const wss = new WebSocketServer({ port: 8080 });
const clients = new Map();

function broadcast(message, exclude = null) {
  const data = JSON.stringify(message);
  wss.clients.forEach((client) => {
    if (client !== exclude && client.readyState === 1) {
      client.send(data);
    }
  });
}

wss.on('connection', (ws) => {
  const userId = Math.random().toString(36).substr(2, 9);
  clients.set(ws, { id: userId, username: `User_${userId}` });
  
  ws.send(JSON.stringify({
    type: 'system',
    message: `Добре дошли! Вашият ID: ${userId}`,
    users: Array.from(clients.values()).map(c => c.username)
  }));
  
  broadcast({
    type: 'system',
    message: `${clients.get(ws).username} се присъедини`
  }, ws);
  
  ws.on('message', (data) => {
    const msg = JSON.parse(data);
    const user = clients.get(ws);
    
    switch (msg.type) {
      case 'chat':
        broadcast({
          type: 'chat',
          user: user.username,
          message: msg.message,
          timestamp: Date.now()
        });
        break;
      case 'setName':
        const oldName = user.username;
        user.username = msg.username;
        broadcast({
          type: 'system',
          message: `${oldName} сега е ${msg.username}`
        });
        break;
    }
  });
  
  ws.on('close', () => {
    const user = clients.get(ws);
    broadcast({
      type: 'system',
      message: `${user.username} напусна`
    });
    clients.delete(ws);
  });
});

console.log('Native WebSocket чат на ws://localhost:8080');


In [ ]:
// chat-server-socketio.js - Socket.IO чат сървър
// npm install socket.io
// Стартирайте: node chat-server-socketio.js

const { Server } = require('socket.io');
const http = require('http');

const httpServer = http.createServer();
const io = new Server(httpServer, {
  cors: { origin: '*' }
});

const users = new Map();

io.on('connection', (socket) => {
  const username = `User_${socket.id.substr(0, 6)}`;
  users.set(socket.id, username);
  
  socket.emit('welcome', {
    message: `Добре дошли ${username}!`,
    users: Array.from(users.values())
  });
  
  socket.broadcast.emit('user joined', { username });
  
  socket.on('chat', (message) => {
    io.emit('chat', {
      user: users.get(socket.id),
      message,
      timestamp: Date.now()
    });
  });
  
  socket.on('setName', (newName) => {
    const oldName = users.get(socket.id);
    users.set(socket.id, newName);
    io.emit('system', { message: `${oldName} сега е ${newName}` });
  });
  
  socket.on('join room', (room) => {
    socket.join(room);
    socket.to(room).emit('system', {
      message: `${users.get(socket.id)} се присъедини към ${room}`
    });
  });
  
  socket.on('room message', ({ room, message }) => {
    io.to(room).emit('chat', {
      user: users.get(socket.id),
      message,
      room,
      timestamp: Date.now()
    });
  });
  
  socket.on('disconnect', () => {
    const username = users.get(socket.id);
    io.emit('user left', { username });
    users.delete(socket.id);
  });
});

httpServer.listen(3000, () => {
  console.log('Socket.IO чат на http://localhost:3000');
});


#### HTML клиент за чат (работи с Native WebSocket сървър)

Запазете следното като `chat-client.html` и отворете в браузър:


In [ ]:
// chat-client.html съдържание:
/*
<!DOCTYPE html>
<html>
<head>
  <title>WebSocket Чат</title>
  <style>
    body { font-family: system-ui, sans-serif; max-width: 600px; margin: 50px auto; padding: 20px; }
    #messages { height: 300px; overflow-y: auto; border: 1px solid #ccc; padding: 10px; margin-bottom: 10px; }
    .system { color: #666; font-style: italic; }
    .chat { margin: 5px 0; }
    .user { font-weight: bold; color: #2196F3; }
    .controls { display: flex; gap: 10px; }
    input { flex: 1; padding: 10px; font-size: 16px; }
    button { padding: 10px 20px; background: #2196F3; color: white; border: none; cursor: pointer; }
    button:hover { background: #1976D2; }
  </style>
</head>
<body>
  <h1>WebSocket Чат</h1>
  <div id="messages"></div>
  <div class="controls">
    <input type="text" id="input" placeholder="Напишете съобщение..." autofocus>
    <button onclick="sendMessage()">Изпрати</button>
  </div>
  
  <script>
    const messages = document.getElementById('messages');
    const input = document.getElementById('input');
    
    const ws = new WebSocket('ws://localhost:8080');
    
    ws.onopen = () => addMessage('Свързан с чат сървъра', 'system');
    ws.onclose = () => addMessage('Изключен от сървъра', 'system');
    
    ws.onmessage = (e) => {
      const data = JSON.parse(e.data);
      if (data.type === 'system') {
        addMessage(data.message, 'system');
      } else if (data.type === 'chat') {
        addMessage(data.user + ': ' + data.message, 'chat');
      }
    };
    
    function addMessage(text, type) {
      const div = document.createElement('div');
      div.className = type;
      div.textContent = text;
      messages.appendChild(div);
      messages.scrollTop = messages.scrollHeight;
    }
    
    function sendMessage() {
      const text = input.value.trim();
      if (!text) return;
      ws.send(JSON.stringify({ type: 'chat', message: text }));
      input.value = '';
    }
    
    input.addEventListener('keypress', (e) => {
      if (e.key === 'Enter') sendMessage();
    });
  </script>
</body>
</html>
*/


## Упражнения

1. **Ехо сървър:** Създайте WebSocket сървър, който връща съобщенията с времева отметка.

2. **Индикатор за писане:** Имплементирайте функция "потребителят пише" с Socket.IO.

3. **Лични съобщения:** Добавете директни съобщения между потребители по socket ID.

4. **Система за присъствие:** Проследявайте онлайн/офлайн статуса и разпращайте промените до всички потребители.

5. **Ограничаване на скоростта:** Имплементирайте ограничение на съобщенията от страна на сървъра (макс. 5 съобщения в секунда).
